# 04 - Unicycle Reference Tracking Control

Load the trained score model from notebook 03 and solve a simple tracking problem with DRGD.

### Setup

Clone/install the repo in Colab and define paths.

In [ ]:
from pathlib import Path
import os
import random
import subprocess
import sys

import numpy as np
import torch

# If the repo is not present in /content, set REPO_URL to your GitHub repo and rerun.
REPO_URL = "https://github.com/<your-org>/score-manifold-optimization.git"
REPO_DIR = Path("/content/score-manifold-optimization")

if not (REPO_DIR / "pyproject.toml").exists():
    if "<" in REPO_URL:
        raise RuntimeError(
            "Repo not found at /content/score-manifold-optimization. "
            "Please set REPO_URL to your repository URL or clone manually."
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_DIR = REPO_DIR / "outputs/unicycle_train_demo"
DATASET_PATH = REPO_DIR / "data/unicycle_T100_n20000.pt"

print(f"Repo: {REPO_DIR}")
print(f"Device: {DEVICE}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Dataset path: {DATASET_PATH}")


### Local Setup (Alternative)

Use this instead when the repo is already cloned on your machine.

In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import torch

start = Path.cwd().resolve()
REPO_DIR = next((p for p in [start, *start.parents] if (p / "pyproject.toml").exists()), None)
if REPO_DIR is None:
    raise RuntimeError("Could not find repo root (missing pyproject.toml in parent dirs).")

os.chdir(REPO_DIR)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_DIR = REPO_DIR / "outputs/unicycle_train_demo"
DATASET_PATH = REPO_DIR / "data/unicycle_T100_n20000.pt"

print(f"Repo root: {REPO_DIR}")
print(f"Device: {DEVICE}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Dataset path: {DATASET_PATH}")

# Install once from terminal (repo root): pip install -e .


### Load Model + Data Context

Load the pretrained score model, diffusion process, train data, and constraint.

In [ ]:
from diffusion.control import DynamicsConstraint
from diffusion.utils.checkpoint_utils import load_pretrained_score_context

context = load_pretrained_score_context(
    checkpoint_dir=CHECKPOINT_DIR,
    dataset_path_override=DATASET_PATH,
    device=DEVICE,
    require_trajectory_data=True,
)

train_data = context.train_data
constraint = context.constraint
model = context.model
diffusion = context.diffusion

if not isinstance(constraint, DynamicsConstraint):
    raise TypeError(f"Expected DynamicsConstraint, got {type(constraint)}")

print(f"Train data shape: {tuple(train_data.shape)}")
print(f"Constraint: {constraint.__class__.__name__}")


### Build Reference and Select Initial Trajectory

Create a simple sine reference in x-y plane and select the closest trajectory from training data at initial trajectory

In [ ]:
from diffusion.control import build_reference_tracking_objective

horizon = constraint.horizon
t = torch.linspace(0.0, 0.5 * torch.pi, horizon, device=train_data.device, dtype=train_data.dtype)
reference = torch.empty(1, horizon, 2)

amp_x = 8.0
amp_y = 3.0
x_ref = amp_x * torch.sin(t)
y_ref = amp_y * torch.sin(2.0 * t)

reference[0, :, 0] = x_ref
reference[0, :, 1] = y_ref
print(f"Reference shape: {tuple(reference.shape)}")

slice_spec = [2,3] #This reference only affects the [2,3] indices of trajectories (i.e. x,y coordinates)
loss_type = "L1" #Loss type for the objective cost


#Take x0 as the trajectory in the train_data with minimal objective value
objective_fn = build_reference_tracking_objective(reference_trajectory=reference, slice_spec=slice_spec, loss_type=loss_type)
with torch.no_grad():
    objective_values = objective_fn(train_data)
    best_idx = int(torch.argmin(objective_values).item())
x0 = train_data[best_idx : best_idx + 1].detach().clone()

#Or select a custom trajectory
#x0 = train_data[[5]].detach().clone()

#Plot the reference and the initial x0\
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(reference[0, :, 0].detach().cpu(), reference[0, :, 1].detach().cpu(), "tab:red", linewidth=2, label="Reference")
ax.plot(x0[...,slice_spec][0, :, 0].detach().cpu(), x0[...,slice_spec][0, :, 1].detach().cpu(), "k", linewidth=2, label="x0")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Unicycle x-y Trajectories")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### Run Tracking (DRGD)

Build the score projector and run `run_reference_tracking`.

In [ ]:
from diffusion.control import build_reference_tracking_objective, run_reference_tracking
from diffusion.optimization import RiemannianConfig, ScoreTangentConfig, build_score_projector
import matplotlib.pyplot as plt

projector = build_score_projector(
    score_model=model,
    diffusion=diffusion,
    score_time=0.25,
    data_dims=train_data.shape[1:],
    n_proj=1,
    tangent_cfg=ScoreTangentConfig(method="vjp"),
)

cfg = RiemannianConfig(
    step_size=1e-2,
    n_steps=1000,
    tangent_mode="project-at-start",
)

def log_fn(x):
    u, y = constraint.split_input_output(x)
    x0_state = y[:, 0]
    y_sim = constraint.system.simulate_out(x0_state, u)
    return {"simulation_mse": float((y_sim - y).pow(2).mean().detach().item())}

tracking = run_reference_tracking(
    reference_trajectory=reference,
    slice_spec=slice_spec,
    projector=projector,
    cfg=cfg,
    x0=x0,
    loss_type=loss_type,
    log_fn=log_fn,
)

print("Tracking complete.")

#Compute some history and plot the objective and feasibility (Simulation MSE) vs. iterations
iterations = range(len(tracking["objective_history"]))
simulation_mse_history = [
    float(x.get("simulation_mse", float("nan"))) if isinstance(x, dict) else float("nan")
    for x in tracking["log_output"]
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(iterations, tracking["objective_history"])
axes[0].set_title("Objective vs Iteration")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Objective")
axes[0].grid(alpha=0.3)

axes[1].plot(iterations, simulation_mse_history)
axes[1].set_title("Simulation MSE vs Iteration")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Simulation MSE")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Plot Trajectories

Plot reference, optimized output, simulated output, and best training trajectory in the x-y plane.

In [ ]:
import matplotlib.pyplot as plt

u_opt, y_opt = constraint.split_input_output(tracking["result"].final_x)
x0_state = y_opt[:, 0]
with torch.no_grad():
    y_sim = constraint.system.simulate_out(x0_state, u_opt)

best_pair = train_data[best_idx : best_idx + 1]
_, y_best = constraint.split_input_output(best_pair)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(reference[0, :, 0].detach().cpu(), reference[0, :, 1].detach().cpu(), "tab:red", linewidth=2, label="Reference")
ax.plot(y_opt[0, :, 0].detach().cpu(), y_opt[0, :, 1].detach().cpu(), color="tab:blue", label="Optimized y")
ax.plot(y_sim[0, :, 0].detach().cpu(), y_sim[0, :, 1].detach().cpu(), color="tab:orange", linestyle="--", label="Simulated y from u_opt")
ax.plot(y_best[0, :, 0].detach().cpu(), y_best[0, :, 1].detach().cpu(), color="g", linestyle=":", alpha=0.9, label="Best training trajectory")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Unicycle x-y Trajectories")
ax.grid(alpha=0.3)
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.show()


### Diagnostics

Print objective improvement, selected initialization index, and simulation tracking error.

In [ ]:
history = tracking["objective_history"]
initial_obj = float(history[0])
final_obj = float(history[-1])

print(f"Initial objective: {initial_obj:.6f}")
print(f"Final objective: {final_obj:.6f}")
print(f"Objective reduction: {initial_obj - final_obj:.6f}")

if tracking["log_output"] and isinstance(tracking["log_output"][-1], dict):
    simulation_mse = tracking["log_output"][-1].get("simulation_mse")
    if simulation_mse is not None:
        print(f"Simulation MSE: {float(simulation_mse):.6e}")
